In [30]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import numpy as np
import time
import os
import csv
import re
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import random

In [37]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

SPLIT_SEED = 42
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

!rm -f vast_english_french.txt

!wget -L "https://raw.githubusercontent.com/David-Ojo/UNCC-ECGR-4106/main/Assignment%204/vast_english_french.txt" -O vast_english_french.txt
FILE_NAME = "vast_english_french.txt"

SOS_TOKEN = "<SOS>"
EOS_TOKEN = "<EOS>"
PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"

MAX_LEN = 10
BATCH_SIZE = 64
EMBED_SIZE = 64
HIDDEN_SIZE = 128
EPOCHS = 100
LEARNING_RATE = 0.001
DROPOUT = 0.00

def set_training_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False



cuda
--2026-07-09 02:24:50--  https://raw.githubusercontent.com/David-Ojo/UNCC-ECGR-4106/main/Assignment%204/vast_english_french.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 42659 (42K) [text/plain]
Saving to: ‘vast_english_french.txt’

vast_english_french 100%[===================>]  41.66K  --.-KB/s    in 0.001s  

2026-07-09 02:24:50 (50.0 MB/s) - ‘vast_english_french.txt’ saved [42659/42659]



In [38]:
# Dataset

def normalize_text(s):
    s = s.lower().strip()
    s = re.sub(r"[^a-zA-ZÀ-ÿ0-9?.!,]+", " ", s)
    return s

pairs = []

with open(FILE_NAME, "r", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split("\t")

        if len(parts) >= 2:
            eng = normalize_text(parts[0])
            fra = normalize_text(parts[1])

            if len(eng.split()) <= MAX_LEN and len(fra.split()) <= MAX_LEN:
                pairs.append((eng, fra))

print("Total sentence pairs:", len(pairs))

train_pairs, val_pairs = train_test_split(
    pairs,
    test_size=0.2,
    random_state=SPLIT_SEED,
    shuffle=True
)

print("Train size:", len(train_pairs))
print("Validation size:", len(val_pairs))


#Vocab Section


def build_vocab(sentences):
    vocab = {
        PAD_TOKEN: 0,
        SOS_TOKEN: 1,
        EOS_TOKEN: 2,
        UNK_TOKEN: 3
    }

    for sentence in sentences:
        for word in sentence.split():
            if word not in vocab:
                vocab[word] = len(vocab)

    return vocab

eng_vocab = build_vocab([p[0] for p in train_pairs])
fra_vocab = build_vocab([p[1] for p in train_pairs])

eng_idx_to_word = {idx: word for word, idx in eng_vocab.items()}
fra_idx_to_word = {idx: word for word, idx in fra_vocab.items()}

PAD_IDX = fra_vocab[PAD_TOKEN]

print("English vocab size:", len(eng_vocab))
print("French vocab size:", len(fra_vocab))

#Dataset - Class

def sentence_to_indices(sentence, vocab):
    indices = [vocab.get(word, vocab[UNK_TOKEN]) for word in sentence.split()]
    indices = [vocab[SOS_TOKEN]] + indices + [vocab[EOS_TOKEN]]

    if len(indices) < MAX_LEN + 2:
        indices += [vocab[PAD_TOKEN]] * ((MAX_LEN + 2) - len(indices))
    else:
        indices = indices[:MAX_LEN + 2]

    return torch.tensor(indices, dtype=torch.long)

class TranslationDataset(Dataset):
    def __init__(self, pairs, eng_vocab, fra_vocab):
        self.pairs = pairs
        self.eng_vocab = eng_vocab
        self.fra_vocab = fra_vocab

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        eng, fra = self.pairs[idx]

        src = sentence_to_indices(eng, self.eng_vocab)
        tgt = sentence_to_indices(fra, self.fra_vocab)

        return src, tgt

train_dataset = TranslationDataset(train_pairs, eng_vocab, fra_vocab)
val_dataset = TranslationDataset(val_pairs, eng_vocab, fra_vocab)

trainloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
valloader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)


Total sentence pairs: 515
Train size: 412
Validation size: 103
English vocab size: 841
French vocab size: 935


In [40]:
#Model 1

class PositionalEncoding(nn.Module):
    def __init__(self, hidden_size, max_len=500):
        super().__init__()

        pe = torch.zeros(max_len, hidden_size)
        position = torch.arange(0, max_len).unsqueeze(1).float()

        div_term = torch.exp(
            torch.arange(0, hidden_size, 2).float()
            * (-np.log(10000.0) / hidden_size)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]


class TransformerSeq2Seq(nn.Module):
    def __init__(
        self,
        src_vocab_size,
        tgt_vocab_size,
        hidden_size,
        num_heads,
        num_layers,
        dropout,
        max_len
    ):
        super().__init__()

        self.src_embedding = nn.Embedding(src_vocab_size, hidden_size, padding_idx=0)
        self.tgt_embedding = nn.Embedding(tgt_vocab_size, hidden_size, padding_idx=0)

        self.src_pos = PositionalEncoding(hidden_size, max_len)
        self.tgt_pos = PositionalEncoding(hidden_size, max_len)

        self.transformer = nn.Transformer(
            d_model=hidden_size,
            nhead=num_heads,
            num_encoder_layers=num_layers,
            num_decoder_layers=num_layers,
            dim_feedforward=hidden_size * 4,
            dropout=dropout,
            batch_first=True
        )

        self.fc_out = nn.Linear(hidden_size, tgt_vocab_size)

    def make_tgt_mask(self, tgt_len):
        return torch.triu(
            torch.ones(tgt_len, tgt_len, device=device),
            diagonal=1
        ).bool()

    def forward(self, src, tgt):
        src_key_padding_mask = src.eq(0)
        tgt_key_padding_mask = tgt.eq(0)

        tgt_mask = self.make_tgt_mask(tgt.size(1))

        src_emb = self.src_pos(self.src_embedding(src))
        tgt_emb = self.tgt_pos(self.tgt_embedding(tgt))

        output = self.transformer(
            src_emb,
            tgt_emb,
            tgt_mask=tgt_mask
            )

        return self.fc_out(output)

        # Optimizer

model = TransformerSeq2Seq(
    src_vocab_size=len(eng_vocab),
    tgt_vocab_size=len(fra_vocab),
    hidden_size=HIDDEN_SIZE,
    num_heads=2,          # Starting configuration
    num_layers=4,         # 2 transformer blocks
    dropout=DROPOUT,
    max_len=MAX_LEN + 2
).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor= 0.1,
    patience=10
)

print(next(model.parameters()).device)

cuda:0


In [41]:

# Training
def train_one_epoch(model, trainloader, optimizer, criterion):
    model.train()
    total_loss = 0.0

    for src, tgt in trainloader:
        src = src.to(device)
        tgt = tgt.to(device)

        decoder_input = tgt[:, :-1]
        target = tgt[:, 1:]

        optimizer.zero_grad()

        outputs = model(src, decoder_input)

        outputs = outputs.reshape(-1, outputs.shape[-1])
        target = target.reshape(-1)

        loss = criterion(outputs, target)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(trainloader)


def evaluate_loss(model, valloader, criterion):
    model.eval()
    total_loss = 0.0

    with torch.no_grad():
        for src, tgt in valloader:
            src = src.to(device)
            tgt = tgt.to(device)

            decoder_input = tgt[:, :-1]
            target = tgt[:, 1:]

            outputs = model(src, decoder_input)

            outputs = outputs.reshape(-1, outputs.shape[-1])
            target = target.reshape(-1)

            loss = criterion(outputs, target)
            total_loss += loss.item()

    return total_loss / len(valloader)

train_losses = []
val_losses = []
best_val_loss = float("inf")
patience = 20
bad_epochs = 0

training_start = time.time()

In [42]:
#Eval/validation


def indices_to_sentence(indices, idx_to_word):
    words = []

    for idx in indices:
        word = idx_to_word.get(int(idx), UNK_TOKEN)

        if word == EOS_TOKEN:
            break

        if word not in [SOS_TOKEN, PAD_TOKEN]:
            words.append(word)

    return " ".join(words)


def translate_sentence(model, sentence, max_len=MAX_LEN + 2):
    model.eval()

    src = sentence_to_indices(sentence, eng_vocab).unsqueeze(0).to(device)

    generated = [fra_vocab[SOS_TOKEN]]

    with torch.no_grad():
        for _ in range(max_len):
            tgt_tensor = torch.tensor([generated], dtype=torch.long, device=device)

            output = model(src, tgt_tensor)

            next_token = output[:, -1, :].argmax(1).item()

            if next_token == fra_vocab[EOS_TOKEN]:
                break

            generated.append(next_token)

    return indices_to_sentence(generated[1:], fra_idx_to_word)


def compute_exact_match_and_bleu(model, val_pairs):
    exact_matches = 0
    bleu_scores = []

    smoothing = SmoothingFunction().method1

    results = []

    for eng, fra in val_pairs:
        predicted = translate_sentence(model, eng)

        reference_tokens = fra.split()
        predicted_tokens = predicted.split()

        exact_match = predicted.strip() == fra.strip()

        if exact_match:
            exact_matches += 1

        bleu = sentence_bleu(
            [reference_tokens],
            predicted_tokens,
            weights=(0.25, 0.25, 0.25, 0.25),
            smoothing_function=smoothing
        )

        bleu_scores.append(bleu)

        results.append({
            "English": eng,
            "Target French": fra,
            "Predicted French": predicted,
            "Exact Match": exact_match,
            "BLEU-4": bleu
        })

    exact_match_accuracy = 100 * exact_matches / len(val_pairs)
    average_bleu = sum(bleu_scores) / len(bleu_scores)

    return exact_match_accuracy, average_bleu, results

exact_acc, avg_bleu, validation_results = compute_exact_match_and_bleu(
    model,
    val_pairs
)

In [43]:

BLOCKS_LIST = [4]
HEADS_LIST = [2]

sweep_results = []

for num_blocks in BLOCKS_LIST:
    for num_heads in HEADS_LIST:

        print(f"\nTraining Transformer | blocks={num_blocks}, heads={num_heads}")

        set_training_seed(42)

        model = TransformerSeq2Seq(
            src_vocab_size=len(eng_vocab),
            tgt_vocab_size=len(fra_vocab),
            hidden_size=HIDDEN_SIZE,
            num_heads=num_heads,
            num_layers=num_blocks,
            dropout=DROPOUT,
            max_len=MAX_LEN + 2
        ).to(device)

        criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
        optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode="min",
            factor=0.1,
            patience=10
        )

        train_losses = []
        val_losses = []

        best_val_loss = float("inf")
        bad_epochs = 0
        patience = 30

        checkpoint_name = f"best_transformer_blocks_{num_blocks}_heads_{num_heads}.pth"

        training_start = time.time()

        for epoch in range(EPOCHS):
            train_loss = train_one_epoch(model, trainloader, optimizer, criterion)
            val_loss = evaluate_loss(model, valloader, criterion)

            train_losses.append(train_loss)
            val_losses.append(val_loss)

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                bad_epochs = 0
                torch.save(model.state_dict(), checkpoint_name)
            else:
                bad_epochs += 1

            scheduler.step(val_loss)

            print(
                f"Blocks {num_blocks}, Heads {num_heads}, "
                f"Epoch [{epoch+1}/{EPOCHS}] "
                f"Train Loss: {train_loss:.4f} "
                f"Val Loss: {val_loss:.4f}"
            )

            if bad_epochs >= patience:
                print(f"Early stopping at epoch {epoch+1}")
                break

        total_training_time = time.time() - training_start

        model.load_state_dict(torch.load(checkpoint_name))
        model.eval()

        exact_acc, avg_bleu, validation_results = compute_exact_match_and_bleu(
            model,
            val_pairs
        )

        sweep_results.append({
            "blocks": num_blocks,
            "heads": num_heads,
            "best_val_loss": best_val_loss,
            "exact_acc": exact_acc,
            "bleu": avg_bleu,
            "training_time": total_training_time,
            "params": sum(p.numel() for p in model.parameters() if p.requires_grad),
            "train_losses": train_losses,
            "val_losses": val_losses
        })

    print(
        f"\nBlocks={num_blocks}, Heads={num_heads} Final Results: "
        f"Best Val Loss: {best_val_loss:.4f}, "
        f"Exact Acc: {exact_acc:.2f}%, "
        f"BLEU-4: {avg_bleu:.4f}, "
        f"Time: {total_training_time:.2f}s"
    )

best_run = max(sweep_results, key=lambda x: x["bleu"])
best_blocks = best_run["blocks"]
best_heads = best_run["heads"]

print("\nBest Run by BLEU-4:")
print(
    f"Blocks {best_blocks}, Heads {best_heads} | "
    f"Best Val Loss: {best_run['best_val_loss']:.4f} | "
    f"Exact Acc: {best_run['exact_acc']:.2f}% | "
    f"BLEU-4: {best_run['bleu']:.4f} | "
    f"Time: {best_run['training_time']:.2f}s"
)
output_dir = "problem3_results"
os.makedirs(output_dir, exist_ok=True)

plt.figure(figsize=(8,5))
plt.plot(best_run["train_losses"], label="Training Loss")
plt.plot(best_run["val_losses"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Cross-Entropy Loss")
plt.title(f"Best Transformer ({best_blocks} blocks, {best_heads} heads)")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(output_dir, f"best_transformer_blocks_{best_blocks}_heads_{best_heads}_loss.png"))
plt.close()


print(f"Validation Exact Match Accuracy: {exact_acc:.2f}%")
print(f"Validation BLEU-4 Score: {avg_bleu:.4f}")
print(f"Total Training Time: {total_training_time:.2f} seconds")

with open(os.path.join(output_dir, "summary.txt"), "w", encoding="utf-8") as f:
    f.write("Problem 1: GRU Encoder-Decoder Baseline\n")
    f.write("=" * 50 + "\n")
    f.write(f"Training size: {len(train_pairs)}\n")
    f.write(f"Validation size: {len(val_pairs)}\n")
    f.write(f"English vocab size: {len(eng_vocab)}\n")
    f.write(f"French vocab size: {len(fra_vocab)}\n")
    f.write(f"Embedding size: {EMBED_SIZE}\n")
    f.write(f"Hidden size: {HIDDEN_SIZE}\n")
    f.write(f"Epochs: {EPOCHS}\n")
    f.write(f"Batch size: {BATCH_SIZE}\n")
    f.write(f"Learning rate: {LEARNING_RATE}\n")
    f.write(f"Final Training Loss: {train_losses[-1]:.4f}\n")
    f.write(f"Final Validation Loss: {val_losses[-1]:.4f}\n")
    f.write(f"Validation Exact Match Accuracy: {exact_acc:.2f}%\n")
    f.write(f"Validation BLEU-4 Score: {avg_bleu:.4f}\n")
    f.write(f"Total Training Time: {total_training_time:.2f} seconds\n")

    sample_results = validation_results[:5]

with open(os.path.join(output_dir, "qualitative_samples.txt"), "w", encoding="utf-8") as f:
    for i, result in enumerate(sample_results, start=1):
        print(f"\nSample {i}")
        print("English:", result["English"])
        print("Target French:", result["Target French"])
        print("Predicted French:", result["Predicted French"])
        print("Exact Match:", result["Exact Match"])
        print(f"BLEU-4: {result['BLEU-4']:.4f}")

        f.write(f"Sample {i}\n")
        f.write(f"English: {result['English']}\n")
        f.write(f"Target French: {result['Target French']}\n")
        f.write(f"Predicted French: {result['Predicted French']}\n")
        f.write(f"Exact Match: {result['Exact Match']}\n")
        f.write(f"BLEU-4: {result['BLEU-4']:.4f}\n")
        f.write("-" * 50 + "\n")

with open(os.path.join(output_dir, "validation_predictions.csv"), "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=[
            "English",
            "Target French",
            "Predicted French",
            "Exact Match",
            "BLEU-4"
        ]
    )

    writer.writeheader()
    writer.writerows(validation_results)


Training Transformer | blocks=4, heads=2
Blocks 4, Heads 2, Epoch [1/100] Train Loss: 6.2773 Val Loss: 5.6488
Blocks 4, Heads 2, Epoch [2/100] Train Loss: 5.7325 Val Loss: 5.4491
Blocks 4, Heads 2, Epoch [3/100] Train Loss: 5.4439 Val Loss: 5.3927
Blocks 4, Heads 2, Epoch [4/100] Train Loss: 5.2613 Val Loss: 5.2937
Blocks 4, Heads 2, Epoch [5/100] Train Loss: 5.0316 Val Loss: 5.1293
Blocks 4, Heads 2, Epoch [6/100] Train Loss: 4.7598 Val Loss: 4.9333
Blocks 4, Heads 2, Epoch [7/100] Train Loss: 4.4258 Val Loss: 4.7234
Blocks 4, Heads 2, Epoch [8/100] Train Loss: 4.1335 Val Loss: 4.5809
Blocks 4, Heads 2, Epoch [9/100] Train Loss: 3.8023 Val Loss: 4.4214
Blocks 4, Heads 2, Epoch [10/100] Train Loss: 3.5713 Val Loss: 4.3944
Blocks 4, Heads 2, Epoch [11/100] Train Loss: 3.3284 Val Loss: 4.2834
Blocks 4, Heads 2, Epoch [12/100] Train Loss: 3.0440 Val Loss: 4.2468
Blocks 4, Heads 2, Epoch [13/100] Train Loss: 2.7968 Val Loss: 4.0777
Blocks 4, Heads 2, Epoch [14/100] Train Loss: 2.5841 Val 